In [1]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("langchain_rag_dataset.txt")
raw_doc = loader.load()
raw_doc

C:\Users\itsar\AppData\Local\Temp\ipykernel_37244\336307699.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\RAG\Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content="LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.\nThe framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.\nMemory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.\nBM25 and vector-based retrieval can be combined in LangChain to support hybrid retrieval strategies.\nFAISS is a high-performance library for similarity search that LangChain 

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_doc)
chunks


[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(metadata={'

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8651.69it/s]


In [7]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, model)
vectorstore

In [8]:
retriver = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 3})
retriver

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020F4137F8C0>, search_type='mmr', search_kwargs={'k': 3})

In [14]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain.chat_models import init_chat_model
llm = init_chat_model(model="groq:openai/gpt-oss-120b")
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.
Context: {context}
Question: {input}
""")


document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever=retriver, combine_docs_chain=document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020F4137F8C0>, search_type='mmr', search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context provided.\nContext: {context}\nQuestion: {input}\n')
            | ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 120B', 'release_d

In [17]:
query = {"input" :"How does LangChain support agents and memory?"}
response = rag_chain.invoke(query)
print(response['answer'])

**LangChain’s support for agents**

* **Tool‑driven agents** – An agent can be given a set of tools (e.g., calculators, web‑search APIs, custom functions, database connectors).  
* **Dynamic tool selection** – The LLM acts as the “brain” that, based on the user’s instruction, decides **which tool to call and in what order** to accomplish the task.  
* **External integration** – Because tools can wrap any external API or database, agents can fetch live data, run computations, or query knowledge bases, greatly extending the capabilities of a pure language model.

**LangChain’s support for memory**

* **ConversationBufferMemory** – Stores the raw transcript of prior turns (user ↔ assistant). Each new LLM call receives the full history, letting the model keep context and produce coherent multi‑turn replies.  
* **ConversationSummaryMemory** – Periodically summarizes the accumulated dialogue into a compact representation. The summary is fed back to the LLM, preserving long‑term context with